# Toronto Nurse Staffing Optimization
**MSBA Prescriptive Analytics · Pepperdine University · Team JCT**

This notebook analyzes nurse staffing demand at Unity Health Toronto using:
- Multiple Linear Regression with COVID-19 and seasonal dummy variables
- Linear Programming optimization across two budget scenarios
- Cost-minimizing headcount recommendation for fiscal year 2025

**Data:** 89 months of actual demand (Oct 2017 – Feb 2025) across 3 nurse qualifications: CCRN, RN, RPN

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Dark theme for all charts
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#0f0f0f',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#888',
    'xtick.color':      '#666',
    'ytick.color':      '#666',
    'text.color':       '#f0f0ee',
    'grid.color':       '#222',
    'legend.frameon':   False,
})

PURPLE = '#8b83e6'
TEAL   = '#2dc8a0'
AMBER  = '#f0a030'
CORAL  = '#e06050'

print('Libraries loaded.')

## 1. Load & Inspect Demand Data

In [ ]:
# Upload 'Nurse_Staffing_Case_-_Team_JCT.xlsx' to Colab before running
# from google.colab import files
# uploaded = files.upload()

xl_path = 'Nurse_Staffing_Case_-_Team_JCT.xlsx'  # update path if needed

demand_df = pd.read_excel(xl_path, sheet_name='Exhibit 2')
demand_df = demand_df[['Month Year','Demand Critical Care RN','Demand RN','Demand RPN','COVID']]
demand_df = demand_df.dropna(subset=['Demand Critical Care RN'])
demand_df['Month Year'] = pd.to_datetime(demand_df['Month Year'])
demand_df.columns = ['Date','CCRN','RN','RPN','COVID']

print(f'Observations: {len(demand_df)}')
print(f'Date range:   {demand_df.Date.min().strftime("%b %Y")} — {demand_df.Date.max().strftime("%b %Y")}')
print(f'COVID months: {demand_df.COVID.sum().astype(int)}')
demand_df[['CCRN','RN','RPN']].describe().round(1)

## 2. Demand Time Series — COVID-19 Impact & Seasonal Patterns

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))

ax.plot(demand_df.Date, demand_df.CCRN, color=PURPLE, lw=2,   label='CCRN (Critical Care RN)')
ax.plot(demand_df.Date, demand_df.RN,   color=TEAL,   lw=2,   label='RN')
ax.plot(demand_df.Date, demand_df.RPN,  color=AMBER,  lw=2,   label='RPN')

# COVID shading
covid = demand_df[demand_df.COVID == 1]
ax.axvspan(covid.Date.min(), covid.Date.max(), alpha=0.12, color=CORAL, zorder=0)
ax.text(covid.Date.min() + pd.Timedelta(days=30), demand_df.CCRN.max() * 0.92,
        '← COVID-19 Surge', color=CORAL, fontsize=10, fontweight='bold')

ax.set_title('Nurse Demand Over Time — Unity Health Toronto (2017–2025)',
             fontsize=14, fontweight='600', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Monthly Hours Demanded')
ax.legend(labelcolor='#aaa', fontsize=10)
plt.tight_layout()
plt.show()

print('Key insight: CCRN demand spiked 3–4x during COVID-19 (Apr 2020 – Sep 2021).')
print(f'Pre-COVID CCRN avg:  {demand_df[demand_df.COVID==0].CCRN.mean():.0f} hrs/mo')
print(f'During-COVID CCRN avg: {demand_df[demand_df.COVID==1].CCRN.mean():.0f} hrs/mo')

## 3. Regression Model Results — Demand Forecasting

In [ ]:
# Regression results from Excel Solver output
reg_results = pd.DataFrame({
    'Nurse Type':     ['CCRN', 'RN', 'RPN'],
    'R²':             [0.849,  0.810, 0.658],
    'Adj. R²':        [0.818,  0.771, 0.588],
    'RMSE (hrs)':     [83.5,   80.1,  101.2],
    'Monthly Trend':  ['+7.6 hrs', '+6.3 hrs', '+5.4 hrs'],
    'COVID Effect':   ['+74.7 hrs', '+34.9 hrs', '-18.9 hrs'],
})
print('Regression Model Summary:')
print(reg_results.to_string(index=False))
print('\nModel: log(1+demand) ~ Month_Index + COVID + Monthly_Seasonality_Dummies')
print('Observations: 77 (89 total minus 12 held for forecasting)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# R² bar chart
ax = axes[0]
nurse_types = ['CCRN', 'RN', 'RPN']
r2     = [0.849, 0.810, 0.658]
adj_r2 = [0.818, 0.771, 0.588]
x = np.arange(3)
w = 0.35

b1 = ax.bar(x - w/2, r2,     w, label='R²',      color=PURPLE, alpha=0.9)
b2 = ax.bar(x + w/2, adj_r2, w, label='Adj. R²', color='#6b63c6', alpha=0.7)
for bar, val in zip(b1, r2):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10)
for bar, val in zip(b2, adj_r2):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, color='#aaa')

ax.axhline(0.8, color=TEAL, linestyle='--', alpha=0.5, linewidth=1.2)
ax.text(2.4, 0.81, 'R²=0.80', color=TEAL, fontsize=8, alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(nurse_types, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Regression Fit by Nurse Type', fontsize=12, fontweight='600', pad=10)
ax.set_ylabel('Goodness of Fit')
ax.legend(labelcolor='#aaa', fontsize=10)

# Monthly trend coefficients
ax2 = axes[1]
trends = [7.576, 6.256, 5.445]
colors = [PURPLE, TEAL, AMBER]
bars = ax2.barh(nurse_types, trends, color=colors, alpha=0.9, height=0.4)
for bar, val in zip(bars, trends):
    ax2.text(val + 0.1, bar.get_y()+bar.get_height()/2,
             f'+{val:.1f} hrs/mo', va='center', fontsize=11, fontweight='600')
ax2.set_title('Monthly Demand Growth Trend\n(Regression Coefficient)', fontsize=12, fontweight='600', pad=10)
ax2.set_xlabel('Hours per Month Increase')
ax2.set_xlim(0, 12)

plt.tight_layout()
plt.show()

## 4. LP Optimization — Cost Comparison Across Scenarios

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Stacked cost bar chart
ax = axes[0]
scenarios = ['Baseline\n(81% avail.)', 'Scenario A\n(75% avail.)', 'Scenario B\n(55% OT prem.)']
regular   = [18.15, 19.28, 18.15]
agency    = [0.43,  0.72,  0.43]
overtime  = [0.79,  0.77,  0.79]
x = np.arange(3)
w = 0.5

ax.bar(x, regular,  w, label='Regular Staff',  color=PURPLE, alpha=0.9)
ax.bar(x, agency,   w, bottom=regular,          label='Agency Staff',   color=CORAL,  alpha=0.9)
ax.bar(x, overtime, w, bottom=[r+a for r,a in zip(regular,agency)],
       label='Overtime', color=AMBER, alpha=0.9)

totals = [r+a+o for r,a,o in zip(regular,agency,overtime)]
for i, t in enumerate(totals):
    ax.text(i, t+0.08, f'${t:.2f}M', ha='center', fontsize=11, fontweight='700')

ax.set_title('Annual Staffing Cost by Scenario', fontsize=12, fontweight='600', pad=10)
ax.set_xticks(x); ax.set_xticklabels(scenarios, fontsize=9)
ax.set_ylabel('Cost ($ Millions)')
ax.set_ylim(0, 22)
ax.legend(labelcolor='#aaa', fontsize=9, loc='upper right')

# Optimal headcount
ax2 = axes[1]
staff  = {'CCRN': 72, 'RN': 86, 'RPN': 51}
colors2 = [PURPLE, TEAL, AMBER]
bars = ax2.bar(staff.keys(), staff.values(), color=colors2, width=0.45, alpha=0.9)
for bar, val in zip(bars, staff.values()):
    ax2.text(bar.get_x()+bar.get_width()/2, val+1.5, str(val),
             ha='center', va='bottom', fontsize=14, fontweight='700')

ax2.set_title('Recommended Headcount (Integer FTE)\nBaseline: $19.35M · RSCR 92.1%',
              fontsize=12, fontweight='600', pad=10)
ax2.set_ylabel('Full-Time Equivalents')
ax2.set_ylim(0, 110)
ax2.text(0.5, 0.10, 'Agency + OT used only\nduring peak demand months',
         transform=ax2.transAxes, ha='center', color=TEAL,
         fontsize=9, fontweight='600',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#0a2a22', edgecolor=TEAL, linewidth=1))

plt.tight_layout()
plt.show()

print('Recommendation: 72 CCRNs, 86 RNs, 51 RPNs')
print('Total annual cost: ~$19.35M – $19.4M')
print('Regular Staff Coverage Ratio: 92.1%')

## 5. Key Findings

| Finding | Detail |
|---------|--------|
| **Regression model fit** | CCRN R²=0.849, RN R²=0.810, RPN R²=0.658 |
| **Long-term growth** | All three nurse types show +5–8 hrs/mo secular increase |
| **COVID-19 impact** | CCRN demand spiked +74.7 hrs/mo during pandemic; RPN unaffected |
| **Optimal headcount** | 72 CCRN + 86 RN + 51 RPN (integer FTE) |
| **Total annual cost** | ~$19.35M – $19.4M across scenarios |
| **Coverage ratio** | 92.1% served by regular staff; remainder via agency/OT at peak months |
| **Key risk** | Model sensitivity to effective availability — dropping below 81% significantly increases agency reliance |

**Course:** MSBA Prescriptive Analytics · Pepperdine University Graziadio Business School  
**Team:** Jules, Cayden, Terence